# Data Summary + Analysis

## Set Up

Import libraries/packages + cleaned data

In [ ]:
# Libraries/packages
import sys

sys.path.append("../")
from src.data_utils import get_feature_lists
from src.config import BASE_PATH
from src.summary_analysis import (
    generate_summary_table,
    get_analysis_df,
    generate_fish_list,
)
import warnings
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
import pandas as pd
from shutil import rmtree

Data

In [ ]:
## BASE
base_data = pd.read_parquet(BASE_PATH / "data" / "raw" / "Cleaned_ORN.parquet")
base_X = base_data.drop("ORN", axis=1)
base_y = base_data["ORN"]

## GET DICTS
DATA_DICT = {
    "base": {"X": base_X, "y": base_y},
}

Re-order a bit

In [ ]:
reordered_cols = [
    ##Pre-Op
    "AGE",
    "BMI",
    "SEX",
    "Diabetes",
    "ASA",
    "PRIOREX",
    "PRECT",
    ## Disease
    "RECUR",
    "SITE",
    "SIZE",
    "LYMPH",
    "STAGE",
    "DEFECT",
    "SECONDPRIMARY",
    ## Surg
    "LENGTH",
    "JEWER",
    "OSTEOTOMY",
    "PLATE",
    "FLAP",
    "TRANSFUS",
    "ISCHEMICTIME",
    "OPTIME",
    ## Immediate post-op
    "REOP",
    "LOHS",
    "POSTCT",
    "WOUNDINF",
    "HGB",
    "ALB",
    ## Long term post-op
    "EXPOSURE",
    "MEDUSED",
    "SURGUSED",
    "PLATETIME",
    "FOLLOWTIME",
    ## Misc
    "RADTIME",
]

for data_type, sub_dict in DATA_DICT.items():
    X_unordered = sub_dict["X"]
    DATA_DICT[data_type]["X"] = X_unordered[reordered_cols].copy()

Classify features by data type

In [ ]:
##Imported func from src
feature_lists = get_feature_lists(DATA_DICT["base"]["X"])
binary_cols = feature_lists["Binary"]
numerical_cols = feature_lists["Numerical"]
nominal_cols = feature_lists["Nominal"]
ordinal_cols = feature_lists["Ordinal"]

Impute

In [ ]:
for data_type, sub_dict in DATA_DICT.items():
    cur_X = sub_dict["X"]
    ## Impute
    imputer = IterativeImputer(
        estimator=None,  # default = BayesianRidge
        initial_strategy="median",
        max_iter=10,
        sample_posterior=False,  # deterministic
    )
    df_impute = cur_X.copy()
    imputed_values = imputer.fit_transform(df_impute[numerical_cols])
    df_impute[numerical_cols] = imputed_values
    ## Ensure no NAs
    assert df_impute.isna().sum().sum() == 0
    DATA_DICT[data_type]["X_imp"] = df_impute

## Summary + Analysis

In [ ]:
# Get features w/ expected freq < 5
fish_dict = generate_fish_list(DATA_DICT, binary_cols, verbose=False)
## Create separate tables for each dataset
final_tables_dict = {}  # Store tables by dataset name


for data_type, sub_dict in DATA_DICT.items():
    og_X = sub_dict["X"]
    X_imp = sub_dict["X_imp"]
    y_df = sub_dict["y"]

    print(f"Working on {data_type}...")

    # Create all_categories specific to THIS dataset
    all_categories = {}
    for col in nominal_cols + binary_cols + ordinal_cols:
        all_categories[col] = X_imp[col].unique()
    ## Get summary
    summary_df = generate_summary_table(
        X_df_final=X_imp,
        X_df_og=og_X,
        outcome_data=y_df,
        data_type=data_type,
        all_categories=all_categories,
        feature_dict=feature_lists,
    )
    # # Df containing univariable values (p-values, ORs w/ CIs)
    analysis_df = get_analysis_df(
        df=X_imp,
        outcome_data=y_df,
        data_type=data_type,
        fish_dict=fish_dict,
    )
    # Verify indices match
    try:
        assert set(analysis_df.index.to_list()) == set(summary_df.index.to_list())
    except AssertionError:
        print(f"Mismatch in {data_type}:")
        print(
            "In analysis but not summary:",
            set(analysis_df.index.to_list()) - set(summary_df.index.to_list()),
        )
        print(
            "In summary but not analysis:",
            set(summary_df.index.to_list()) - set(analysis_df.index.to_list()),
        )
        raise AssertionError(
            f"Analysis and summary tables DO NOT match for {data_type}!"
        )
    # Join summary and analysis for this dataset
    final_table = summary_df.join(analysis_df, how="left").fillna("NA")

    # Store in dictionary
    final_tables_dict[data_type] = final_table
    print(f"Completed {data_type} table with shape {final_table.shape}")

Export

In [ ]:
## Set up path
export_path = BASE_PATH / "results" / "tables" / "summary_analysis"
if export_path.exists():
    rmtree(export_path)
    warnings.warn(f"Over-writing folder at path {export_path}")
export_path.mkdir(exist_ok=True, parents=True)
## Export
base_table = final_tables_dict["base"]
base_table.to_excel(export_path / "base.xlsx", index=True)